In [5]:
import pandas as pd
df=pd.read_csv("https://raw.githubusercontent.com/Saranmanoharan511/customer-propensity-model/main/customer-propensity-model/cleaned_data.zip", compression="zip")


In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 392692 entries, 0 to 392691
Data columns (total 9 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   InvoiceNo    392692 non-null  int64  
 1   StockCode    392692 non-null  object 
 2   Description  392692 non-null  object 
 3   Quantity     392692 non-null  int64  
 4   InvoiceDate  392692 non-null  object 
 5   UnitPrice    392692 non-null  float64
 6   CustomerID   392692 non-null  float64
 7   Country      392692 non-null  object 
 8   TotalPrice   392692 non-null  float64
dtypes: float64(3), int64(2), object(4)
memory usage: 27.0+ MB


In [7]:
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])

In [8]:
snapshot_date=df['InvoiceDate'].max()+pd.Timedelta(days=1)

In [13]:
df.groupby("CustomerID")


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,TotalPrice
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom,15.30
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom,22.00
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34
...,...,...,...,...,...,...,...,...,...
392556,581578,22556,PLASTERS IN TIN CIRCUS PARADE,12,2011-12-09 12:16:00,1.65,12713.0,Germany,19.80
392557,581578,22976,CIRCUS PARADE CHILDRENS EGG CUP,12,2011-12-09 12:16:00,1.25,12713.0,Germany,15.00
392558,581578,23255,CHILDRENS CUTLERY CIRCUS PARADE,12,2011-12-09 12:16:00,4.15,12713.0,Germany,49.80
392559,581578,84997D,CHILDRENS CUTLERY POLKADOT PINK,8,2011-12-09 12:16:00,4.15,12713.0,Germany,33.20


In [35]:
rfm = df.groupby('CustomerID').agg({
    'InvoiceDate': lambda x: (snapshot_date - x.max()).days,
    'InvoiceNo': 'count',
    'TotalPrice': 'sum'
})

rfm.columns = ['Recency', 'Frequency', 'Monetary']

In [15]:
rfm.info()

<class 'pandas.core.frame.DataFrame'>
Index: 4338 entries, 12346.0 to 18287.0
Data columns (total 3 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   Recency    4338 non-null   int64  
 1   Frequency  4338 non-null   int64  
 2   Monetary   4338 non-null   float64
dtypes: float64(1), int64(2)
memory usage: 135.6 KB


In [36]:
rfm

,Recency,Frequency,Monetary
CustomerID,,,
12346.0,326,1,77183.60
12347.0,2,182,4310.00
12348.0,75,31,1797.24
12349.0,19,73,1757.55
12350.0,310,17,334.40
...,...,...,...
18280.0,278,10,180.60
18281.0,181,7,80.82
18282.0,8,12,178.05


In [37]:
rfm["AvgOrderValue"]=rfm['Monetary']/rfm['Frequency']

In [38]:
df = df.sort_values(['CustomerID', 'InvoiceDate'])

df['prev_date'] = df.groupby('CustomerID')['InvoiceDate'].shift(1)
df['days_between'] = (df['InvoiceDate'] - df['prev_date']).dt.days

interval = df.groupby('CustomerID')['days_between'].mean()

rfm['AvgPurchaseInterval'] = interval

In [39]:
rfm

,Recency,Frequency,Monetary,AvgOrderValue,AvgPurchaseInterval
CustomerID,,,,,
12346.0,326,1,77183.60,77183.600000,NaN
12347.0,2,182,4310.00,23.681319,2.000000
12348.0,75,31,1797.24,57.975484,9.400000
12349.0,19,73,1757.55,24.076027,0.000000
12350.0,310,17,334.40,19.670588,0.000000
...,...,...,...,...,...
18280.0,278,10,180.60,18.060000,0.000000
18281.0,181,7,80.82,11.545714,0.000000
18282.0,8,12,178.05,14.837500,10.727273


Nan and 0 shows that single day purchase or one time buyers


In [41]:
rfm[rfm['AvgPurchaseInterval']==0]

,Recency,Frequency,Monetary,AvgOrderValue,AvgPurchaseInterval
CustomerID,,,,,
12349.0,19,73,1757.55,24.076027,0.0
12350.0,310,17,334.40,19.670588,0.0
12353.0,204,4,89.00,22.250000,0.0
12354.0,232,58,1079.40,18.610345,0.0
12355.0,214,13,459.40,35.338462,0.0
...,...,...,...,...,...
18276.0,44,14,335.86,23.990000,0.0
18277.0,58,8,110.38,13.797500,0.0
18278.0,74,9,173.90,19.322222,0.0


In [42]:
rfm=rfm.fillna(0)

In [43]:
rfm

,Recency,Frequency,Monetary,AvgOrderValue,AvgPurchaseInterval
CustomerID,,,,,
12346.0,326,1,77183.60,77183.600000,0.000000
12347.0,2,182,4310.00,23.681319,2.000000
12348.0,75,31,1797.24,57.975484,9.400000
12349.0,19,73,1757.55,24.076027,0.000000
12350.0,310,17,334.40,19.670588,0.000000
...,...,...,...,...,...
18280.0,278,10,180.60,18.060000,0.000000
18281.0,181,7,80.82,11.545714,0.000000
18282.0,8,12,178.05,14.837500,10.727273


In [44]:
rfm.reset_index(inplace=True)

from google.colab import files
rfm.to_csv('rfm_features.csv', index=False)
files.download('rfm_features.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>